# Tutorial: Kernel Performance Distance between two sets of MIDI performances

In this tutorial, we will use Pereval's command-line interface to measure dissimilarity between two sets of expressive MIDI piano performances. Specifically, we will calculate **Kernel Performance Distance** (KPD), which uses contextual embeddings to reveal performance-level differences.

**The goals of this tutorial are:**
- Familiarize the reader with the command-line tools and modules of Pereval, a toolkit for evaluation of MIDI piano performances
- Learn how to construct metadata for MIDI paths to provide input to Pereval tools
- Learn the intuition behind Kernel Performance Distance and how to apply it

**Prerequisites:**
- Before running this notebook, please install the Pereval Python package and Aria following [these instructions](https://realfolkcode.github.io/pereval/installation.html)
- Please also download the PianoCoRe MIDI dataset from [Zenodo](https://zenodo.org/records/19186016). Only the refined subset is required (6 GB) 

## Import

In [1]:
import pandas as pd
import os
import json
import torch
from sklearn.model_selection import train_test_split
from functools import partial

In [2]:
from pereval.feature_extractor import FeatureExtractor
from pereval.kmd import KernelPerformanceDistance

## Preparing metadata

In this section, we will learn how to construct a dictionary of MIDI filepaths. Each dictionary will be associated with a set of MIDI performances. Then we can pass them as input arguments to Pereval's CLI for extracting the contextual embeddings.

We will use [PianoCoRe (TISMIR 2026, Borovik)](https://zenodo.org/records/19186016), a symbolic dataset of piano music. PianoCoRe contains high quality scores and performances of classical music, which will be useful for our educational purposes. Please follow the instructions in the dataset link and download `PianoCoRe-1.0-refined.zip` as well as `metadata.csv`.

In [ ]:
# The path to the PianoCoRe metadata (metadata.csv)
METADATA_PATH = ...

# The path to the PianoCoRe refined directory (PianoCoRe/refined)
PIANOCORE_DIR = ...

In [4]:
# First, let's load the metadata
metadata = pd.read_csv(METADATA_PATH)

# Here, we select a high-quality subset of PianoCoRe
metadata = metadata[metadata["tier_a_star"]]

# Now let's have a look at the metadata
metadata

,id,composer,composition,movement,performance_id,split,tier_b,tier_a,tier_a_star,score_dataset,...,is_refined,refined_score_midi_path,refined_score_note_count,refined_score_duration,refined_performance_midi_path,refined_performance_note_count,refined_performance_interpolated_note_count,refined_performance_duration,refined_alignment_path,refined_recall
41,PianoCoRe_000041,"Albéniz,_Isaac",Angustia,NaN,Aria_052064_0,train,True,True,True,PDMX,...,True,"Albéniz,_Isaac/Angustia/score_PDMX_refined.mid",1107.0,138.000,"Albéniz,_Isaac/Angustia/Aria_052064_0_refined.mid",1107.0,88.0,148.7594,"Albéniz,_Isaac/Angustia/Aria_052064_0_refined_...",0.9205
42,PianoCoRe_000042,"Albéniz,_Isaac",Angustia,NaN,Aria_575833_0,train,True,True,True,PDMX,...,True,"Albéniz,_Isaac/Angustia/score_PDMX_refined.mid",1107.0,138.000,"Albéniz,_Isaac/Angustia/Aria_575833_0_refined.mid",1107.0,91.0,147.3802,"Albéniz,_Isaac/Angustia/Aria_575833_0_refined_...",0.9178
43,PianoCoRe_000043,"Albéniz,_Isaac",Angustia,NaN,Aria_672738_0,train,True,True,True,PDMX,...,True,"Albéniz,_Isaac/Angustia/score_PDMX_refined.mid",1107.0,138.000,"Albéniz,_Isaac/Angustia/Aria_672738_0_refined.mid",1107.0,89.0,131.0125,"Albéniz,_Isaac/Angustia/Aria_672738_0_refined_...",0.9196
44,PianoCoRe_000044,"Albéniz,_Isaac",Angustia,NaN,PERiScoPe_432953202,train,True,True,True,PDMX,...,True,"Albéniz,_Isaac/Angustia/score_PDMX_refined.mid",1107.0,138.000,"Albéniz,_Isaac/Angustia/PERiScoPe_432953202_re...",1107.0,98.0,166.4844,"Albéniz,_Isaac/Angustia/PERiScoPe_432953202_re...",0.9115
123,PianoCoRe_000123,"Albéniz,_Isaac","España,_Op.165",2._Tango._Andantino,Aria_035594_0,train,True,True,True,MuseScore,...,True,"Albéniz,_Isaac/España,_Op.165/2._Tango._Andant...",664.0,93.000,"Albéniz,_Isaac/España,_Op.165/2._Tango._Andant...",664.0,56.0,163.2385,"Albéniz,_Isaac/España,_Op.165/2._Tango._Andant...",0.9157
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250041,PianoCoRe_250041,"Zipoli,_Domenico",Fughetta_in_E_minor,NaN,Aria_159968_0,train,True,True,True,PDMX,...,True,"Zipoli,_Domenico/Fughetta_in_E_minor/score_PDM...",219.0,27.375,"Zipoli,_Domenico/Fughetta_in_E_minor/Aria_1599...",219.0,2.0,66.5104,"Zipoli,_Domenico/Fughetta_in_E_minor/Aria_1599...",0.9909
250042,PianoCoRe_250042,"Zipoli,_Domenico",Fughetta_in_E_minor,NaN,Aria_301389_0,train,True,True,True,PDMX,...,True,"Zipoli,_Domenico/Fughetta_in_E_minor/score_PDM...",219.0,27.375,"Zipoli,_Domenico/Fughetta_in_E_minor/Aria_3013...",219.0,2.0,48.8500,"Zipoli,_Domenico/Fughetta_in_E_minor/Aria_3013...",0.9909
250043,PianoCoRe_250043,"Zipoli,_Domenico",Fughetta_in_E_minor,NaN,Aria_651335_0,train,True,True,True,PDMX,...,True,"Zipoli,_Domenico/Fughetta_in_E_minor/score_PDM...",219.0,27.375,"Zipoli,_Domenico/Fughetta_in_E_minor/Aria_6513...",219.0,2.0,45.6708,"Zipoli,_Domenico/Fughetta_in_E_minor/Aria_6513...",0.9909
250044,PianoCoRe_250044,"Zipoli,_Domenico",Fughetta_in_E_minor,NaN,Aria_772751_0,train,True,True,True,PDMX,...,True,"Zipoli,_Domenico/Fughetta_in_E_minor/score_PDM...",219.0,27.375,"Zipoli,_Domenico/Fughetta_in_E_minor/Aria_7727...",219.0,3.0,50.7896,"Zipoli,_Domenico/Fughetta_in_E_minor/Aria_7727...",0.9863


KPD reflects expressive differences between sets better if the musical pieces in the sets are represented by multiple interpretations. Each row in the metadata corresponds to a performance, which we will aggregate by a composer, a composition, and a movement.

In [5]:
# Let's aggregate the metadata by compositions
valid_pieces = (metadata.groupby(['composer', 'composition', 'movement']).size()).reset_index()

# Since PianoCoRe is large scale, we select a small subset for demonstration
# We select pieces with at least four and no more than twenty performances
valid_pieces = valid_pieces.rename(columns={0: "num_performances"})
valid_pieces = valid_pieces[valid_pieces["num_performances"] >= 4][valid_pieces["num_performances"] <= 20]
# We also subsample 10% of the filtered pieces (feel free to adjust this percentage)
valid_pieces = valid_pieces.sample(frac=0.1)
valid_pieces

/tmp/ipykernel_231933/274889425.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  valid_pieces = valid_pieces[valid_pieces["num_performances"] >= 4][valid_pieces["num_performances"] <= 20]


,composer,composition,movement,num_performances
160,"Bach,_Johann_Sebastian","Partita_No.4_in_D_major,_BWV_828",7._Gigue,17
158,"Bach,_Johann_Sebastian","Partita_No.4_in_D_major,_BWV_828",5._Sarabande,19
737,"Haydn,_Joseph","Piano_Sonata_No.52,_Keyboard_Sonata_in_G_major...",1._Allegro_con_brio_(G_major),20
187,"Bach,_Johann_Sebastian","The_Well-Tempered_Clavier,_Book_I,_BWV_846-869","No.14_in_F_sharp_minor,_BWV_859:_Prelude",18
162,"Bach,_Johann_Sebastian","Partita_No.5_in_G_major,_BWV_829",2._Allemande,11
91,"Bach,_Johann_Sebastian","English_Suite_No.6_in_D_minor,_BWV_811",2._Allemande,10
391,"Beethoven,_Ludwig_van","Piano_Sonata_No.2_in_A_major,_Op.2_No.2",3._Scherzo_(Allegretto),6
1180,"Tchaikovsky,_Pyotr_Ilyich","The_Sleeping_Beauty_(ballet),_Op.66",Act_1._6._Grande_valse_villageoise,8
716,"Grieg,_Edvard","Peer_Gynt_Suite_No.1,_Op.46",4._In_the_Hall_of_the_Mountain_King,15
681,"Elgar,_Edward","Variations_on_an_Original_Theme_'Enigma',_Op.36","Variation_IX._Adagio_""Nimrod""",16


In [6]:
# Now let's combine performance filepaths into a list for each piece 
df = pd.merge(metadata, valid_pieces, on=['composer', 'composition', 'movement'])
df = df.groupby(['composer', 'composition', 'movement'])["refined_performance_midi_path"].agg(list)
df

composer                   composition                                                 movement                                                  
Bach,_Johann_Sebastian     9_Kleine_Präludien,_BWV_924-932                             7._Prelude_in_G_minor,_BWV_930                                [Bach,_Johann_Sebastian/9_Kleine_Präludien,_BW...
                           English_Suite_No.6_in_D_minor,_BWV_811                      2._Allemande                                                  [Bach,_Johann_Sebastian/English_Suite_No.6_in_...
                                                                                       6._Gavotte_I-II                                               [Bach,_Johann_Sebastian/English_Suite_No.6_in_...
                           French_Suite_No.5_in_G_major,_BWV_816                       5._Bourree                                                    [Bach,_Johann_Sebastian/French_Suite_No.5_in_G...
                           Partita_No.4_in_D_major,_BWV_82

In [7]:
# In order to use Pereval, we need a dictionary, where keys are score IDs and values are lists of performance filepaths
# Let's "compress" the composer, composition, and movement information into score identifiers
df = df.reset_index()["refined_performance_midi_path"]
df

0     [Bach,_Johann_Sebastian/9_Kleine_Präludien,_BW...
1     [Bach,_Johann_Sebastian/English_Suite_No.6_in_...
2     [Bach,_Johann_Sebastian/English_Suite_No.6_in_...
3     [Bach,_Johann_Sebastian/French_Suite_No.5_in_G...
4     [Bach,_Johann_Sebastian/Partita_No.4_in_D_majo...
5     [Bach,_Johann_Sebastian/Partita_No.4_in_D_majo...
6     [Bach,_Johann_Sebastian/Partita_No.5_in_G_majo...
7     [Bach,_Johann_Sebastian/The_Well-Tempered_Clav...
8     [Bach,_Johann_Sebastian/The_Well-Tempered_Clav...
9     [Bach,_Johann_Sebastian/The_Well-Tempered_Clav...
10    [Bach,_Johann_Sebastian/The_Well-Tempered_Clav...
11    [Bach,_Johann_Sebastian/The_Well-Tempered_Clav...
12    [Bartók,_Béla/For_Children,_Sz.42/Volume_2._18...
13    [Beethoven,_Ludwig_van/Piano_Sonata_No.2_in_A_...
14    [Dvořák,_Antonín/Poetic_Tone_Pictures,_Op.85/1...
15    [Elgar,_Edward/Variations_on_an_Original_Theme...
16    [Grieg,_Edvard/Peer_Gynt_Suite_No.1,_Op.46/4._...
17    [Haydn,_Joseph/Piano_Sonata_No.52,_Keyboar

In [8]:
# Let's also not forget about prepending the paths with the root directory of PianoCoRe
df = df.apply(lambda perf_lst : [os.path.join(PIANOCORE_DIR, perf_path) for perf_path in perf_lst])

We are almost done with the metadata. The next step is to split it into two parts of roughly the same size. These two dictionaries will be associated with the two sets for which we will calculate Kernel Performance Distance. In the following cells, these sets are named **A** and **B**.

In [9]:
df = df.apply(partial(train_test_split, test_size=0.5))
df_a = df.apply(lambda lst: lst[0])
df_b = df.apply(lambda lst: lst[1])

Now let's check the sizes of these sets (the number of performances for each score in a set).

In [10]:
df_a.apply(len)

0      9
1      5
2      2
3     10
4      9
5      8
6      5
7      9
8      9
9      9
10    10
11    10
12     3
13     3
14     8
15     8
16     7
17    10
18     2
19     6
20     6
21    10
22     9
23     4
Name: refined_performance_midi_path, dtype: int64

In [11]:
df_b.apply(len)

0     10
1      5
2      2
3     10
4     10
5      9
6      6
7      9
8      9
9     10
10    10
11    10
12     3
13     3
14     9
15     8
16     8
17    10
18     3
19     7
20     6
21    10
22     9
23     4
Name: refined_performance_midi_path, dtype: int64

Finally, let's save the dictionaries as .json files (the format that Pereval expects).

In [12]:
EVALUATION_PATHS_A = "./evaluation_path_dict_a.json"
EVALUATION_PATHS_B = "./evaluation_path_dict_b.json"

df_a = df_a.to_dict()
df_b = df_b.to_dict()

with open(EVALUATION_PATHS_A, "w", encoding="utf8") as f:
    f.write(json.dumps(df_a, indent=2))

with open(EVALUATION_PATHS_B, "w", encoding="utf8") as f:
    f.write(json.dumps(df_b, indent=2))

## Calculating Aria embeddings

We are now ready to encode MIDI performances with [Aria](https://github.com/EleutherAI/aria), a symbolic music understanding model. To do this, we will run a Pereval script. Let's take a closer look at the `pereval-embed-aria` util to understand the format of its arguments:

In [13]:
!pereval-embed-aria --help

usage: pereval-embed-aria [-h] --checkpoint_path CHECKPOINT_PATH
                          [--evaluation_path EVALUATION_PATH]
                          [--output_path OUTPUT_PATH]

options:
  -h, --help            show this help message and exit
  --checkpoint_path CHECKPOINT_PATH
                        The path to aria model checkpoint
  --evaluation_path EVALUATION_PATH
                        A directory with performances or a .json dictionary
                        with performance paths
  --output_path OUTPUT_PATH
                        The output path to store embeddings in a .pt format.


Please make sure you have already installed the [Aria](https://github.com/EleutherAI/aria) package and downloaded the `aria-medium-embedding` checkpoint.

In the `--evaluation_path` field, we will use the .json files that we created in the earlier section. You might have noticed that this argument can also be a directory. This is an alternative option, which might be more convenient if the scores and performances are following [this structure convention](https://realfolkcode.github.io/pereval/cli.html#midi-path-convention). However, a .json file enables the usage of arbitrary filepaths and is less restricted.

The `--output_path` field indicates where Aria embeddings will be saved (should be a .pt file).

In [ ]:
# The path to the Aria-medium-embedding checkpoint (model.safetensors) 
CHECKPOINT_PATH = ...

# Feel free to also change these variables. By default, the embeddings will be saved in the notebook's directory.
OUTPUT_PATH_A = "./aria_embeddings_a.pt"
OUTPUT_PATH_B = "./aria_embeddings_b.pt"

Encoding MIDI files with Aria should take around one minute on a GPU.

In [15]:
!pereval-embed-aria \
    --checkpoint_path $CHECKPOINT_PATH \
    --evaluation_path $EVALUATION_PATHS_A \
    --output_path $OUTPUT_PATH_A

100%|██████████████████████████████████████████| 24/24 [00:48<00:00,  2.03s/it]


In [16]:
!pereval-embed-aria \
    --checkpoint_path $CHECKPOINT_PATH \
    --evaluation_path $EVALUATION_PATHS_B \
    --output_path $OUTPUT_PATH_B

100%|██████████████████████████████████████████| 24/24 [00:50<00:00,  2.09s/it]


If everything is good, running the above commands should have yielded progress bars and the *.pt files located in `OUTPUT_PATH_A` and `OUTPUT_PATH_B`.

## Calculating Kernel Performance Distance

Now let's use the `FeatureExtractor` class from Pereval to load the embeddings. It returns a NumPy array of embeddings as well as score IDs and performance paths.

In [17]:
feature_extractor = FeatureExtractor()

emb_a, score_lst_a, perf_lst_a = feature_extractor.extract_features(OUTPUT_PATH_A)
emb_b, score_lst_b, perf_lst_b = feature_extractor.extract_features(OUTPUT_PATH_B)

emb_a = torch.from_numpy(emb_a)
emb_b = torch.from_numpy(emb_b)

At its core, KPD is simply kernelized pairwise distances between embeddings aggregated by compositions. The distances capture the pairwise dissimilarities between the sets **A** and **B** and also the dissimilarities *within* the sets. Therefore, through KPD we can assess how similarly compositions are performed in **A** and **B**.

Estimating KPD requires setting the bandwidth for Gaussian kernel beforehand. Since the bandwidth is calculated on a *reference* set, this makes KPD **asymmetric**. So when we calculate KPD, it means we measure how dissimilar a *query* set is with respect to a *reference* set.

In [ ]:
# Instantiate KPD estimator from Pereval
kpd_estimator = KernelPerformanceDistance()

# Calculate the bandwidth on a reference set A
kpd_estimator.calculate_bandwidth(
    ref_emb=emb_a,
    label_lst=score_lst_a)

kpd = kpd_estimator.compute_kpd(
    x=emb_a,
    y=emb_b,
    x_label_lst=score_lst_a,
    y_label_lst=score_lst_b
)

print("Kernel Performance Distance (reference set A):", kpd)

Kernel Performance Distance (reference set A): 0.2716636657714844


In some cases, KPD could be negative. This might be counterintuitive because we normally expect distances to be nonnegative. However, we *estimate* KPD, which is a random variable itself, and this estimation is **unbiased**. Consequently, the estimation of KPD can fluctuate around zero when the sets are too similar. In our example, **A** and **B** are random subsets from the same corpus sampled without any assumption. So we expect KPD to be small.

In [19]:
# Now let's switch the roles of the sets A and B and calculate KPD again

kpd_estimator = KernelPerformanceDistance()

kpd_estimator.calculate_bandwidth(
    ref_emb=emb_b,
    label_lst=score_lst_b)

kpd = kpd_estimator.compute_kpd(
    x=emb_a,
    y=emb_b,
    x_label_lst=score_lst_a,
    y_label_lst=score_lst_b
)

print("Kernel Performance Distance (reference set B):", kpd)

Kernel Performance Distance (reference set B): 0.2670402526855469


Changing the reference set should not result in drastic changes in KPD. Since **A** and **B** are similar, the bandwidth estimation in both cases should lead to roughly the same results. 